# Finite Difference Methods

Finite Difference Methods (FDM) price options by replacing the continuous Black–Scholes PDE with calculations on a grid of **stock prices** and **time points**.

This notebook focuses on the **Explicit Finite Difference Method** and ends with **Bilinear Interpolation**.


## 1. Explicit Finite Difference Method

The basic steps are:

1. Create a grid of stock prices and times.
2. Set the option payoff at expiration.
3. Work backward through the grid to calculate option values.
4. Interpolate when the required stock price or time lies between grid points.

The explicit method is easy to implement, but the grid must be sufficiently fine for numerical stability.


### Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)


### Specify Parameters


In [ ]:
# Option parameters
T = 1.0          # time to maturity
E = 100          # strike price
r = 0.05         # risk-free rate
vol = 0.20       # volatility

# Grid parameters
NAS = 20         # number of asset-price steps
call = True      # True = call, False = put
american = False # False = European option


### Create the Explicit FDM Grid

Let $E$ be the strike price, $N_S$ the number of stock-price steps (`NAS`), and $\sigma$ the volatility (`vol`). The grid extends from zero to $2E$:

$$
\Delta S = \frac{2E}{N_S}, \qquad S_i = i\Delta S, \qquad i=0,\ldots,N_S.
$$

The code chooses a preliminary time step, then adjusts it to divide the maturity exactly:

$$
\Delta t_{\mathrm{trial}} = \frac{0.9}{\sigma^2 N_S^2}.
$$

$$
N_T = \left\lfloor \frac{T}{\Delta t_{\mathrm{trial}}} \right\rfloor + 1,
\qquad \Delta t = \frac{T}{N_T}.
$$

The columns store time to maturity $\tau_k=T-k\Delta t$. Write $V_i^k=V(S_i,\tau_k)$; the final column has $\tau_{N_T}=0$.

This is the step-selection rule used by the code; numerical stability also depends on the full finite-difference coefficients.


In [ ]:
# Grid sizes
ds = 2 * E / NAS

dt = 0.9 / (vol**2 * NAS**2)
NTS = int(T / dt) + 1
dt = T / NTS

# Stock-price points
s = np.arange(0, (NAS + 1) * ds, ds)

# Time-to-maturity points: T down to 0
t = T - np.arange(NTS * dt, -dt, -dt)[::-1]

# Empty option-value grid
grid = pd.DataFrame(
    np.zeros((len(s), len(t))),
    index=s,
    columns=np.round(t, 3)
)

print("Asset step (ds):", ds)
print("Time step (dt):", dt)
print("Grid shape:", grid.shape)


### Set the Payoff at Expiration

For a European call:

$$
C(S,0)=\max(S-E,0).
$$

For a European put:

$$
P(S,0)=\max(E-S,0).
$$

The code combines these using $f=1$ for a call and $f=-1$ for a put:

$$
V_i^{N_T}=\max\bigl(f(S_i-E),0\bigr).
$$


In [ ]:
# +1 for call, -1 for put
flag = 1 if call else -1

# Expiration is the last column (time to maturity = 0)
payoff = np.maximum(flag * (s - E), 0)
grid.iloc[:, -1] = payoff

grid


### Fill the Grid

For calendar time $t$, the Black–Scholes PDE (with no dividends) is

$$
\frac{\partial V}{\partial t}
+\frac{1}{2}\sigma^2 S^2\frac{\partial^2 V}{\partial S^2}
+rS\frac{\partial V}{\partial S}-rV=0.
$$

At each interior point, the code uses centered differences from the already calculated column:

$$
\Delta_i^{k+1}\approx
\frac{V_{i+1}^{k+1}-V_{i-1}^{k+1}}{2\Delta S}.
$$

$$
\Gamma_i^{k+1}\approx
\frac{V_{i+1}^{k+1}-2V_i^{k+1}+V_{i-1}^{k+1}}{(\Delta S)^2}.
$$

Calendar-time Theta is

$$
\Theta_i^{k+1}
=-\frac{1}{2}\sigma^2 S_i^2\Gamma_i^{k+1}
-rS_i\Delta_i^{k+1}+rV_i^{k+1}.
$$

Since time to maturity is $\tau=T-t$, the backward pricing update is

$$
V_i^k=V_i^{k+1}-\Delta t\,\Theta_i^{k+1}.
$$

The boundary rules implemented in the code are

$$
V_0^k=e^{-r\Delta t}V_0^{k+1},
\qquad V_{N_S}^k=2V_{N_S-1}^k-V_{N_S-2}^k.
$$

The upper boundary uses linear extrapolation. If American exercise is enabled, each updated value is compared with immediate exercise:

$$
V_i^k\leftarrow\max\left(V_i^k,\max\bigl(f(S_i-E),0\bigr)\right).
$$


In [ ]:
for k in range(len(t) - 2, -1, -1):

    for i in range(1, len(s) - 1):

        # First derivative with respect to stock price
        delta = (
            grid.iloc[i + 1, k + 1]
            - grid.iloc[i - 1, k + 1]
        ) / (2 * ds)

        # Second derivative with respect to stock price
        gamma = (
            grid.iloc[i + 1, k + 1]
            - 2 * grid.iloc[i, k + 1]
            + grid.iloc[i - 1, k + 1]
        ) / ds**2

        # Black-Scholes PDE
        theta = (
            -0.5 * vol**2 * s[i]**2 * gamma
            - r * s[i] * delta
            + r * grid.iloc[i, k + 1]
        )

        # Move one step backward
        grid.iloc[i, k] = grid.iloc[i, k + 1] - dt * theta

    # Boundary at S = 0
    grid.iloc[0, k] = grid.iloc[0, k + 1] * np.exp(-r * dt)

    # Boundary at large S
    grid.iloc[-1, k] = (
        2 * grid.iloc[-2, k]
        - grid.iloc[-3, k]
    )

    # Optional early-exercise condition
    if american:
        intrinsic = np.maximum(flag * (s - E), 0)
        grid.iloc[:, k] = np.maximum(grid.iloc[:, k], intrinsic)

grid = grid.round(2)
grid


### Compare Payoff and Current Option Value


In [ ]:
option_values = pd.DataFrame({
    "Stock Price": s,
    "Payoff": payoff,
    "Option Value": grid.iloc[:, 0].to_numpy()
})

option_values


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(s, payoff, label="Payoff")
plt.plot(s, grid.iloc[:, 0], label="Option Value")
plt.xlabel("Stock Price")
plt.ylabel("Value")
plt.title("Explicit FDM: Payoff and Option Value")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 2. Put the Calculation into a Function

The same calculation can be placed in one function so that different strikes, volatilities, maturities, and grid sizes can be tested easily.


In [ ]:
def efdm_option(E=100, vol=0.20, r=0.05, T=1,
                NAS=20, call=False, american=False):

    # Step 1: Create grid
    ds = 2 * E / NAS

    dt = 0.9 / (vol**2 * NAS**2)
    NTS = int(T / dt) + 1
    dt = T / NTS

    s = np.arange(0, (NAS + 1) * ds, ds)
    t = T - np.arange(NTS * dt, -dt, -dt)[::-1]

    values = pd.DataFrame(
        np.zeros((len(s), len(t))),
        index=s,
        columns=np.round(t, 3)
    )

    # Step 2: Set payoff
    flag = 1 if call else -1
    values.iloc[:, -1] = np.maximum(flag * (s - E), 0)

    # Step 3: Work backward
    for k in range(len(t) - 2, -1, -1):

        for i in range(1, len(s) - 1):

            delta = (
                values.iloc[i + 1, k + 1]
                - values.iloc[i - 1, k + 1]
            ) / (2 * ds)

            gamma = (
                values.iloc[i + 1, k + 1]
                - 2 * values.iloc[i, k + 1]
                + values.iloc[i - 1, k + 1]
            ) / ds**2

            theta = (
                -0.5 * vol**2 * s[i]**2 * gamma
                - r * s[i] * delta
                + r * values.iloc[i, k + 1]
            )

            values.iloc[i, k] = (
                values.iloc[i, k + 1] - dt * theta
            )

        values.iloc[0, k] = (
            values.iloc[0, k + 1] * np.exp(-r * dt)
        )

        values.iloc[-1, k] = (
            2 * values.iloc[-2, k]
            - values.iloc[-3, k]
        )

        if american:
            intrinsic = np.maximum(flag * (s - E), 0)
            values.iloc[:, k] = np.maximum(
                values.iloc[:, k],
                intrinsic
            )

    return values.round(2)


In [ ]:
# Example: European put
fdm_grid = efdm_option(
    E=100,
    vol=0.20,
    r=0.05,
    T=1,
    NAS=20,
    call=False,
    american=False
)

fdm_grid


## 3. Bilinear Interpolation

The FDM grid gives option values only at specific stock prices and times to maturity. For a point between grid nodes, use the four surrounding values.

Let $S_1\leq S\leq S_2$ and $\tau_1\leq\tau\leq\tau_2$, and define

$$
Q_{11}=V(S_1,\tau_1),\qquad Q_{12}=V(S_1,\tau_2),
$$

$$
Q_{21}=V(S_2,\tau_1),\qquad Q_{22}=V(S_2,\tau_2).
$$

The normalized position within the rectangle is

$$
u=\frac{S-S_1}{S_2-S_1},\qquad
v=\frac{\tau-\tau_1}{\tau_2-\tau_1}.
$$

The bilinear interpolation formula is

$$
\begin{aligned}
V(S,\tau)\approx {}& (1-u)(1-v)Q_{11}+u(1-v)Q_{21}\\
&+(1-u)vQ_{12}+uvQ_{22}.
\end{aligned}
$$

If only one coordinate lies between grid points, the code uses linear interpolation in that coordinate. For example, at a fixed stock-price node:

$$
V(S_1,\tau)\approx (1-v)Q_{11}+vQ_{12}.
$$

At an exact grid point, it returns the stored value directly.


In [ ]:
def bilinear_interpolation(asset_price, ttm, df):

    stock = df.index.to_numpy(dtype=float)
    time = df.columns.to_numpy(dtype=float)

    # Check that the requested point is inside the grid
    if not (stock.min() <= asset_price <= stock.max()):
        raise ValueError("Asset price is outside the grid.")

    if not (time.min() <= ttm <= time.max()):
        raise ValueError("Time to maturity is outside the grid.")

    # Surrounding stock-price points
    s1 = stock[stock <= asset_price].max()
    s2 = stock[stock >= asset_price].min()

    # Surrounding time points
    t1 = time[time <= ttm].max()
    t2 = time[time >= ttm].min()

    # Exact grid point
    if s1 == s2 and t1 == t2:
        return float(df.loc[s1, t1])

    # Only time interpolation is needed
    if s1 == s2:
        v1 = df.loc[s1, t1]
        v2 = df.loc[s1, t2]
        return v1 + (ttm - t1) * (v2 - v1) / (t2 - t1)

    # Only stock-price interpolation is needed
    if t1 == t2:
        v1 = df.loc[s1, t1]
        v2 = df.loc[s2, t1]
        return v1 + (asset_price - s1) * (v2 - v1) / (s2 - s1)

    # Four surrounding option values
    Q11 = df.loc[s1, t1]
    Q12 = df.loc[s1, t2]
    Q21 = df.loc[s2, t1]
    Q22 = df.loc[s2, t2]

    # Bilinear interpolation
    value = (
        Q11 * (s2 - asset_price) * (t2 - ttm)
        + Q21 * (asset_price - s1) * (t2 - ttm)
        + Q12 * (s2 - asset_price) * (ttm - t1)
        + Q22 * (asset_price - s1) * (ttm - t1)
    ) / ((s2 - s1) * (t2 - t1))

    return float(value)


In [ ]:
# Example 1
value_1 = bilinear_interpolation(
    asset_price=90,
    ttm=0.25,
    df=fdm_grid
)

print("Estimated option value:", round(value_1, 4))


In [ ]:
# Example 2
value_2 = bilinear_interpolation(
    asset_price=115,
    ttm=0.25,
    df=fdm_grid
)

print("Estimated option value:", round(value_2, 4))
